# The value of information: should Scotia wait to learn the weather?

This continues the Scotia Snowboard newsvendor problem (costs $c = 200$, price $p = 480$, salvage $s = 80$, with the same good/poor weather mixture for demand). Now we ask a different question: **what is it worth to learn the weather before committing to a production quantity?**

We compare two decision problems on the *same* demand scenarios:

- **One unconditional quantity** — Scotia must pick a single production level before the weather is known, exactly as before.
- **Two weather-conditional quantities** — Scotia learns the weather first, then produces a quantity tailored to that state.

The gap between the two expected profits is the **expected value of perfect information** (EVPI) about the weather: the most Scotia should be willing to pay for a perfectly accurate seasonal forecast.

## Setup

Imports and a fixed random seed. We use `minimize_scalar` for the single-quantity problem and `minimize` for the two-quantity problem.

In [1]:
import numpy as np
from scipy.optimize import minimize_scalar
from scipy.optimize import minimize
np.random.seed(666)

## Two decision problems on the same scenarios

We reuse the weather-dependent demand draws, keeping each scenario's weather label so the conditional policy can act on it. First we re-solve for the single best unconditional quantity; then we optimize a separate quantity for each weather state. The difference in expected profit is the EVPI.

In [2]:
production_cost  = 200
selling_price    = 480
discounted_price = 80

# Weather mixture: index 0 = good (high demand, p=1/3), index 1 = poor (low demand, p=2/3)
demand_prob = [1/3, 2/3]
demand_distributions = [
    {"mean": 100_000, "std_dev": 35_000},
    {"mean":  75_000, "std_dev": 22_500},
]

def calculate_profit(produced, demand, production_cost, selling_price, discounted_price):
    sold   = min(produced, demand)
    unsold = max(0, produced - demand)
    revenue = sold * selling_price + unsold * discounted_price
    cost    = produced * production_cost
    return revenue - cost

def sample_demand_and_weather(demand_prob, demand_distributions):
    weather = np.random.choice(len(demand_prob), p=demand_prob)
    distribution = demand_distributions[weather]
    demand = np.random.normal(distribution["mean"], distribution["std_dev"])
    return round(demand), weather

# Draw scenarios once, keeping each scenario's weather state for the conditional policy
n_simulations = 10_000
demand_scenarios = []
weather_scenarios = []
for _ in range(n_simulations):
    demand, weather = sample_demand_and_weather(demand_prob, demand_distributions)
    demand_scenarios.append(demand)
    weather_scenarios.append(weather)

# Profit when production may depend on the realized weather state
def monte_carlo_expected_profit(produced, demand_scenarios, weather_scenarios, production_cost, selling_price, discounted_price):
    profits = []
    for i, demand in enumerate(demand_scenarios):
        production = produced[weather_scenarios[i]]
        profits.append(calculate_profit(production, demand, production_cost, selling_price, discounted_price))
    return np.mean(profits)

# One unconditional quantity: same production in both weather states
def objective_single(produced):
    return -monte_carlo_expected_profit([produced, produced], demand_scenarios, weather_scenarios, production_cost, selling_price, discounted_price)

# Two quantities: one per weather state
def objective_two(produced):
    return -monte_carlo_expected_profit(produced, demand_scenarios, weather_scenarios, production_cost, selling_price, discounted_price)

result_single = minimize_scalar(objective_single, bounds=(0, 200_000), method='bounded')
optimal_production_single = round(result_single.x)
max_profit_single = round(-result_single.fun)

print("Single production level decision problem:")
print("Optimal number of snowboards to produce:", optimal_production_single)
print("Expected profit:", max_profit_single)

initial_guess = [optimal_production_single, optimal_production_single]
bounds = [(0, 200_000), (0, 200_000)]
result_two = minimize(objective_two, initial_guess, bounds=bounds)
optimal_production_two = [round(x) for x in result_two.x]
max_profit_two = round(-result_two.fun)

print("\nTwo production levels decision problem (conditional on weather):")
print("Optimal production for weather = 0 (good, high demand):", optimal_production_two[0])
print("Optimal production for weather = 1 (poor, low demand): ", optimal_production_two[1])
print("Expected profit:", max_profit_two)

# Expected value of perfect information about the weather
profit_difference = max_profit_two - max_profit_single
print("\nDifference in expected profits (value of knowing the weather):", profit_difference)

Single production level decision problem:
Optimal number of snowboards to produce: 95999
Expected profit: 19157489

Two production levels decision problem (conditional on weather):
Optimal production for weather = 0 (good, high demand): 116870
Optimal production for weather = 1 (poor, low demand):  87459
Expected profit: 19631501

Difference in expected profits (value of knowing the weather): 474012
